# اختبار عربي محلول لتجربة SmartGrade
خمسة أسئلة علوم من30 درجة، ومفتاح تصحيح، وست أوراق إجابة مطبوعة بصيغتي PNG وPDF.
شغّل الخلية الأولى لاستخراج الملفات، والثانية لتنزيل الحزمة. لا تحتاج مفاتيح API أو تثبيت مكتبات.
ابدأ بصورة student-c.png: درجاتها المرجعية للأسئلة 2،5،4،3،3، والإجمالي17/30.
أنشئ اختبارًا جديدًا في مشروع SmartGrade من exam-spec.json أو teacher-key.md، ثم ارفع الصورة.
راجع OCR وحدد إجابة كل سؤال واحفظها قبل طلب اقتراح الدرجات. النموذج يحتاج إعداد مزوده مسبقًا.
هذه الخلايا تنزّل ملفات فقط؛ لا تنشئ بيانات داخل مؤسستك ولا تستدعي مزود تصحيح.
راجع README.md؛ الدرجات مرجعية لهذه العينات وليست قياسًا معتمدًا لدقة النموذج.


In [ ]:
# @title 1. استخراج ملفات الاختبار المضمنة
import base64, hashlib, io, stat, zipfile
from pathlib import Path, PurePosixPath
BUNDLE_SHA256 = "434e815593ac7a56c724909886287b120802ae0b4fbc06e50645c6317650c364"
BUNDLE_B64 = "..."
blob = base64.b64decode(BUNDLE_B64, validate=True)
if hashlib.sha256(blob).hexdigest() != BUNDLE_SHA256:
    raise ValueError("بصمة ملفات الاختبار غير صحيحة")
TARGET = Path("/content/SmartGrade-Arabic-Essay-Test")
TARGET.mkdir(parents=True, exist_ok=True)
if TARGET.is_symlink():
    raise ValueError("اختر مجلدًا غير رمزي")
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    for member in archive.infolist():
        part = PurePosixPath(member.filename)
        if part.is_absolute() or ".." in part.parts or stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError("مسار غير آمن")
        destination = TARGET.joinpath(*part.parts)
        if TARGET.resolve() not in destination.resolve().parents:
            raise ValueError("مسار خارج مجلد الاختبار")
        if member.file_size > 10 * 1024 * 1024:
            raise ValueError("ملف أكبر من الحد المحدد")
        content = archive.read(member)
        if destination.exists() and destination.read_bytes() != content:
            raise FileExistsError("ملف معدل موجود؛ اختر مجلدًا جديدًا دون استبداله")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
ZIP_FILE = TARGET / "SmartGrade-Arabic-Essay-Test.zip"
ZIP_FILE.write_bytes(blob)
print("ملفات الاختبار جاهزة:", TARGET)


In [ ]:
# @title 2. تنزيل الاختبار والمفتاح وجميع الصور
from google.colab import files
files.download(str(ZIP_FILE))


In [ ]:
# @title 3. تنزيل صورة مفردة للرفع — اختياري
CASE = "student-c" # @param ["student-a", "student-b", "student-c", "student-d", "student-e", "student-f"]
DOWNLOAD_ONE_IMAGE = False # @param {type:"boolean"}
if DOWNLOAD_ONE_IMAGE:
    from google.colab import files
    files.download(str(TARGET / (CASE + ".png")))
